In [2]:
import pandas as pd
from collections import Counter, defaultdict
import re
import os

In [2]:
# Load the Excel file
df_anwani = pd.read_excel("/kaggle/input/konkani-pos-tagged-reference-sentences/anwani_bigru_100.xlsx")
df_dhyas = pd.read_excel("/kaggle/input/konkani-pos-tagged-reference-sentences/dhyaskonkani_bigru_100.xlsx")
df_ILCI_75 = pd.read_excel("/kaggle/input/konkani-pos-tagged-reference-sentences/75_biGRU.xlsx")

In [3]:
## Use this for 1L sentences
folder_path = '/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences'
files = os.listdir(folder_path)
txt_files = [file for file in files if file.endswith('.txt')]
len(txt_files)

100

In [4]:
taggedData = []
for file in txt_files:
    with open("/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences/" + file, 'r', encoding='utf-8') as file:
        content = file.read()
    content = content.split("\n")
    content = content[1:]  # Skip the first line
    taggedDataFile = [line.split('\t')[1] for line in content if 'Value' not in line.split('\t')[1]]
    taggedData.append(taggedDataFile)
    
allTaggedData = [item for sublist in taggedData for item in sublist]
allTaggedData = list(set(allTaggedData))
len(allTaggedData)

99986

In [7]:
df_ILCI = pd.DataFrame({'ref_sentences': allTaggedData})
df_ILCI.head()

,ref_sentences
0,मूळ\N_NN हिन्दी\N_NNP पुस्तकाक\N_NN पानांची\N_...
1,पूण\CC_CCS आमी\PR_PRP सेरीकल्चर\N_NN फार्मांत\...
2,मतींत\N_NN दवरचें\V_VM_VF जितली\QT_QTF कूड\N_N...
3,ह्या\DM_DMD पयशिल्या\JJ दुर्मीळ\JJ दोंगरांनीय\...
4,सुमार\QT_QTF बाव्वीस\QT_QTC किमी\N_NN चलून\V_V...


In [3]:
import ast

def extract_sent(tup_arr):
    # Handle missing values
    if pd.isna(tup_arr):
        return ""
    
    # Convert string to list if needed
    if isinstance(tup_arr, str):
        try:
            tup_arr = ast.literal_eval(tup_arr)
        except Exception as e:
            print(f"Error parsing string: {tup_arr} -> {e}")
            return ""
    
    # Ensure it's a list of tuples
    if isinstance(tup_arr, list):
        try:
            words = [word for word, tag in tup_arr]
            sentence = " ".join(words)
            return sentence
        except Exception as e:
            print(f"Error extracting words from: {tup_arr} -> {e}")
            return ""
    
    # If it’s none of the above
    return ""

#extract_sent([('ग्रेटर', 'N_NN'), ('नोयडा', 'N_NN'), ('वेस्टाचे', 'N_NN'), ('त्रास', 'N_NN'), ('कमी', 'QT_QTF'), ('जावपाचें', 'V_VM_VNF'), ('नांवूच', 'N_NN'), ('घेना', 'V_VM_VF'), ('.', 'RD_PUNC')])

df_ILCI_75["sentences"] = df_ILCI_75["predicted_sentences"].apply(extract_sent)
df_ILCI_75.head()

,ref_sentences,predicted_sentences,accuracy,sentences
0,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...,"[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), (...",100.000000,सहस्त्रधारा ही देहरादूनाची पळोवपाची मुखेल सुवात .
1,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...,"[('खजियाराची', 'N_NNP'), ('सांजवेळ', 'N_NN'), ...",100.000000,खजियाराची सांजवेळ दर भोंवडेकाराक थाकाय दिता .
2,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...,"[('आधुनिकीकरण', 'N_NN'), ('आनी', 'CC_CCD'), ('...",100.000000,आधुनिकीकरण आनी नागरीकरणाक लागून जिणेशैलेंत जाल...
3,हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\QT_QTC ऑक...,"[('हे', 'DM_DMD'), ('मागणे', 'N_NN'), ('प्रमाण...",95.000000,"हे मागणे प्रमाणें ३१ ऑक्टोबर , २०००चे मध्य रात..."
4,स्पोर्टिंगाक\N_NN सुरवातीक\N_NST गोल\N_NN करपा...,"[('स्पोर्टिंगाक', 'N_NN'), ('सुरवातीक', 'N_NN'...",91.304348,स्पोर्टिंगाक सुरवातीक गोल करपाच्यो कांय संदी ल...


In [4]:
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

df_ILCI_75["output"] = df_ILCI_75["predicted_sentences"].apply(makeTagedSentences)
df_ILCI_75.head()

,ref_sentences,predicted_sentences,accuracy,sentences,output
0,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...,"[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), (...",100.000000,सहस्त्रधारा ही देहरादूनाची पळोवपाची मुखेल सुवात .,सहस्त्रधारा\N_NNP ही\DM_DMD देहरादूनाची\N_NNP ...
1,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...,"[('खजियाराची', 'N_NNP'), ('सांजवेळ', 'N_NN'), ...",100.000000,खजियाराची सांजवेळ दर भोंवडेकाराक थाकाय दिता .,खजियाराची\N_NNP सांजवेळ\N_NN दर\JJ भोंवडेकाराक...
2,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...,"[('आधुनिकीकरण', 'N_NN'), ('आनी', 'CC_CCD'), ('...",100.000000,आधुनिकीकरण आनी नागरीकरणाक लागून जिणेशैलेंत जाल...,आधुनिकीकरण\N_NN आनी\CC_CCD नागरीकरणाक\N_NN लाग...
3,हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\QT_QTC ऑक...,"[('हे', 'DM_DMD'), ('मागणे', 'N_NN'), ('प्रमाण...",95.000000,"हे मागणे प्रमाणें ३१ ऑक्टोबर , २०००चे मध्य रात...",हे\DM_DMD मागणे\N_NN प्रमाणें\PSP ३१\QT_QTC ऑक...
4,स्पोर्टिंगाक\N_NN सुरवातीक\N_NST गोल\N_NN करपा...,"[('स्पोर्टिंगाक', 'N_NN'), ('सुरवातीक', 'N_NN'...",91.304348,स्पोर्टिंगाक सुरवातीक गोल करपाच्यो कांय संदी ल...,स्पोर्टिंगाक\N_NN सुरवातीक\N_NN गोल\N_NN करपाच...


In [10]:
# Regular expression to match word_tag format (e.g., "गोंयांत_N_NNP")
#pattern = re.compile(r"(.+?)_([A-Z_]+)")

In [22]:
# Initialize frequency counter and tag mapping
word_freq = Counter()
word_tags = defaultdict(set)

# Process each tagged sentence
for sentence in df_anwani["ref_sentences"].dropna():
    tokens = sentence.strip().split()
    for token in tokens:
        if "_" not in token:
            continue  # Skip untagged tokens
        try:
            word, tag = token.rsplit("\\", 1)  # Split only at the last underscore
            word_freq[word] += 1
            word_tags[word].add(tag)
        except ValueError:
            continue  # In case of malformed token

# Get top 50 most frequent words
top_50 = word_freq.most_common(600)

# Prepare structured data
data = []
for word, freq in top_50:
    tags = ", ".join(sorted(word_tags[word]))
    data.append((word, freq, tags))

# Create and display DataFrame
result_df1 = pd.DataFrame(data, columns=["Word", "Frequency", "Tags_ref"])
result_df1.tail(10)

,Word,Frequency,Tags_ref
546,खंयच,1,DM_DMI
547,घराच्या,1,N_NN
548,चार,1,QT_QTC
549,वणटींनी,1,N_NN
550,आसतलीं,1,V_VM_VNF
551,तीं,1,PR_PRP
552,न्हय,1,RP_NEG
553,धडपड,1,N_NN
554,नशिबान,1,N_NN
555,चलप,1,V_VM_VNG


In [23]:
# Initialize frequency counter and tag mapping
word_freq = Counter()
word_tags = defaultdict(set)

# Process each tagged sentence
for sentence in df_anwani["output"].dropna():
    tokens = sentence.strip().split()
    for token in tokens:
        if "_" not in token:
            continue  # Skip untagged tokens
        try:
            word, tag = token.rsplit("\\", 1)  # Split only at the last underscore
            word_freq[word] += 1
            word_tags[word].add(tag)
        except ValueError:
            continue  # In case of malformed token

# Get top 50 most frequent words
top_50 = word_freq.most_common(600)

# Prepare structured data
data = []
for word, freq in top_50:
    tags = ", ".join(sorted(word_tags[word]))
    data.append((word, freq, tags))

# Create and display DataFrame
result_df2 = pd.DataFrame(data, columns=["Word", "Frequency", "Tags_op"])
result_df2.tail(10)

,Word,Frequency,Tags_op
544,घराच्या,1,N_NN
545,चार,1,QT_QTC
546,वणटींनी,1,N_NN
547,आसतलीं,1,V_VM_VNF
548,तीं,1,PR_PRP
549,काटकसरीची,1,N_NN
550,न्हय,1,RP_NEG
551,धडपड,1,N_NN
552,नशिबान,1,N_NN
553,चलप,1,V_VNG


In [24]:
result_df_anwani = pd.merge(result_df1, result_df2, on='Word', how='inner')
result_df_anwani.drop(columns=['Frequency_y'], inplace=True)
result_df_anwani.rename(columns={'Frequency_x': 'Word_Freq'}, inplace=True)
result_df_anwani


,Word,Word_Freq,Tags_ref,Tags_op
0,.,199,RD_PUNC,RD_PUNC
1,?,46,RD_PUNC,RD_PUNC
2,ना,43,RP_NEG,"RP_NEG, V_VAUX_VF"
3,",",20,RD_PUNC,RD_PUNC
4,जाल्यार,16,"CC_CCS, V_VM_VNF",CC_CCS
...,...,...,...,...
532,तीं,1,PR_PRP,PR_PRP
533,न्हय,1,RP_NEG,RP_NEG
534,धडपड,1,N_NN,N_NN
535,नशिबान,1,N_NN,N_NN


In [45]:
result_df.to_excel("word_freq_anwani.xlsx", index=False)

In [61]:
# Function to count tags in a column and return a DataFrame
def count_tags_df(df, column, col_name='Tag_Frequency'):
    tag_counts = {}
    for tags in df[column].dropna():
        for tag in map(str.strip, tags.split(',')):
            tag_counts[tag] = tag_counts.get(tag, 0) + 1
    return pd.DataFrame(tag_counts.items(), columns=['Tag', col_name]).sort_values(by=col_name, ascending=False)

# Compute tag frequency DataFrames
df_ref_tag_freq = count_tags_df(result_df, 'Tags_ref', col_name='Ref_Frequency')
df_op_tag_freq = count_tags_df(result_df, 'Tags_op', col_name='Op_Frequency')

# Display results
print("Tag Frequencies in Tags_ref:")
df_ref_tag_freq

Tag Frequencies in Tags_ref:


,Tag,Ref_Frequency
5,N_NN,262
3,V_VM_VF,48
13,V_VM_VNF,39
4,N_NNP,36
11,QT_QTC,19
8,DM_DMD,14
12,N_NST,11
2,V_VAUX_VF,10
6,QT_QTF,10
7,CC_CCS,6


In [62]:
print("\nTag Frequencies in Tags_op:")
df_op_tag_freq


Tag Frequencies in Tags_op:


,Tag,Op_Frequency
5,N_NN,259
3,V_VM_VF,50
13,V_VM_VNF,42
4,N_NNP,35
10,QT_QTC,20
8,DM_DMD,14
12,N_NST,11
6,QT_QTF,10
2,V_VAUX_VF,8
14,PR_PRP,5


In [25]:
# Initialize frequency counter and tag mapping
word_freq = Counter()
word_tags = defaultdict(set)

# Process each tagged sentence
for sentence in df_dhyas["ref_sentences"].dropna():
    tokens = sentence.strip().split()
    for token in tokens:
        if "_" not in token:
            continue  # Skip untagged tokens
        try:
            word, tag = token.rsplit("\\", 1)  # Split only at the last underscore
            word_freq[word] += 1
            word_tags[word].add(tag)
        except ValueError:
            continue  # In case of malformed token

# Get top 50 most frequent words
top_50 = word_freq.most_common(600)

# Prepare structured data
data = []
for word, freq in top_50:
    tags = ", ".join(sorted(word_tags[word]))
    data.append((word, freq, tags))

# Create and display DataFrame
result_df1 = pd.DataFrame(data, columns=["Word", "Frequency", "Tags_ref"])
result_df1.tail(10)

,Word,Frequency,Tags_ref
481,सकाळीं,1,N_NN
482,माडाचेर,1,N_NN
483,चडटात,1,V_VM_VF
484,माडाक,1,N_NN
485,बांदिल्लें,1,V_VM_VNF
486,सकयल,1,N_NST
487,देंवयता,1,V_VM_VF
488,जाल्लो,1,V_VM_VNF
489,माडाची,1,N_NN
490,पितात,1,V_VM_VF


In [26]:
# Initialize frequency counter and tag mapping
word_freq = Counter()
word_tags = defaultdict(set)

# Process each tagged sentence
for sentence in df_dhyas["output"].dropna():
    tokens = sentence.strip().split()
    for token in tokens:
        if "_" not in token:
            continue  # Skip untagged tokens
        try:
            word, tag = token.rsplit("\\", 1)  # Split only at the last underscore
            word_freq[word] += 1
            word_tags[word].add(tag)
        except ValueError:
            continue  # In case of malformed token

# Get top 50 most frequent words
top_50 = word_freq.most_common(600)

# Prepare structured data
data = []
for word, freq in top_50:
    tags = ", ".join(sorted(word_tags[word]))
    data.append((word, freq, tags))

# Create and display DataFrame
result_df2 = pd.DataFrame(data, columns=["Word", "Frequency", "Tags_op"])
result_df2.tail(10)

,Word,Frequency,Tags_op
482,सकाळीं,1,N_NN
483,माडाचेर,1,N_NN
484,चडटात,1,V_VM_VNF
485,माडाक,1,N_NN
486,बांदिल्लें,1,V_VM_VNF
487,सकयल,1,N_NST
488,देंवयता,1,V_VM_VF
489,जाल्लो,1,V_VM_VNF
490,माडाची,1,N_NN
491,पितात,1,V_VM_VNF


In [27]:
result_df_dhyas = pd.merge(result_df1, result_df2, on='Word', how='inner')
result_df_dhyas.drop(columns=['Frequency_y'], inplace=True)
result_df_dhyas.rename(columns={'Frequency_x': 'Word_Freq'}, inplace=True)
result_df_dhyas

,Word,Word_Freq,Tags_ref,Tags_op
0,.,93,RD_PUNC,RD_PUNC
1,",",19,RD_PUNC,RD_PUNC
2,आनी,16,CC_CCD,CC_CCD
3,आसता,12,"V_VAUX_VF, V_VM_VF","V_VAUX_VF, V_VM_VF"
4,गोंयांत,12,N_NNP,N_NNP
...,...,...,...,...
477,सकयल,1,N_NST,N_NST
478,देंवयता,1,V_VM_VF,V_VM_VF
479,जाल्लो,1,V_VM_VNF,V_VM_VNF
480,माडाची,1,N_NN,N_NN


In [5]:
# Initialize frequency counter and tag mapping
word_freq = Counter()
word_tags = defaultdict(set)

# Process each tagged sentence
for sentence in df_ILCI_75["ref_sentences"].dropna():
    tokens = sentence.strip().split()
    for token in tokens:
        if "_" not in token:
            continue  # Skip untagged tokens
        try:
            word, tag = token.rsplit("\\", 1)  # Split only at the last underscore
            word_freq[word] += 1
            word_tags[word].add(tag)
        except ValueError:
            continue  # In case of malformed token

# Get top 50 most frequent words
top_50 = word_freq.most_common(800)

# Prepare structured data
data = []
for word, freq in top_50:
    tags = ", ".join(sorted(word_tags[word]))
    data.append((word, freq, tags))

# Create and display DataFrame
result_df1 = pd.DataFrame(data, columns=["Word", "Frequency", "Tags_ref"])
result_df1.tail(10)

,Word,Frequency,Tags_ref
637,छाजपूरा,1,N_NNP
638,तालरा,1,N_NNP
639,रातीचो,1,N_NN
640,दोंगर,1,N_NN
641,सुर्याचीं,1,N_NN
642,किरणां,1,N_NN
643,पडटात,1,V_VM_VF
644,चान्न्याचे,1,N_NN
645,राती,1,N_NN
646,देखाव,1,N_NN


In [6]:
# Initialize frequency counter and tag mapping
word_freq = Counter()
word_tags = defaultdict(set)

# Process each tagged sentence
for sentence in df_ILCI_75["output"].dropna():
    tokens = sentence.strip().split()
    for token in tokens:
        if "_" not in token:
            continue  # Skip untagged tokens
        try:
            word, tag = token.rsplit("\\", 1)  # Split only at the last underscore
            word_freq[word] += 1
            word_tags[word].add(tag)
        except ValueError:
            continue  # In case of malformed token

# Get top 50 most frequent words
top_50 = word_freq.most_common(800)

# Prepare structured data
data = []
for word, freq in top_50:
    tags = ", ".join(sorted(word_tags[word]))
    data.append((word, freq, tags))

# Create and display DataFrame
result_df2 = pd.DataFrame(data, columns=["Word", "Frequency", "Tags_op"])
result_df2.tail(10)

,Word,Frequency,Tags_op
630,छाजपूरा,1,N_NNP
631,तालरा,1,N_NNP
632,रातीचो,1,N_NN
633,दोंगर,1,N_NN
634,सुर्याचीं,1,N_NN
635,किरणां,1,N_NN
636,पडटात,1,V_VM_VF
637,चान्न्याचे,1,N_NN
638,राती,1,N_NN
639,देखाव,1,N_NN


In [7]:
result_df_ilci_test = pd.merge(result_df1, result_df2, on='Word', how='inner')
result_df_ilci_test.drop(columns=['Frequency_y'], inplace=True)
result_df_ilci_test.rename(columns={'Frequency_x': 'Word_Freq'}, inplace=True)
result_df_ilci_test

,Word,Word_Freq,Tags_ref,Tags_op
0,.,73,RD_PUNC,RD_PUNC
1,आनी,28,CC_CCD,CC_CCD
2,",",24,RD_PUNC,RD_PUNC
3,आसा,7,V_VM_VF,V_VM_VF
4,ह्या,7,"DM_DMD, PR_PRP",DM_DMD
...,...,...,...,...
628,किरणां,1,N_NN,N_NN
629,पडटात,1,V_VM_VF,V_VM_VF
630,चान्न्याचे,1,N_NN,N_NN
631,राती,1,N_NN,N_NN


In [8]:
result_df_ilci_test.to_excel("word_freq_ILCI_test.xlsx", index=False)

In [9]:
# Initialize frequency counter and tag mapping
word_freq = Counter()
word_tags = defaultdict(set)

# Process each tagged sentence
for sentence in df_ILCI["ref_sentences"].dropna():
    tokens = sentence.strip().split()
    for token in tokens:
        if "_" not in token:
            continue  # Skip untagged tokens
        try:
            word, tag = token.rsplit("\\", 1)  # Split only at the last underscore
            word_freq[word] += 1
            word_tags[word].add(tag)
        except ValueError:
            continue  # In case of malformed token

# Get top 50 most frequent words
top_50 = word_freq.most_common(1000)

# Prepare structured data
data = []
for word, freq in top_50:
    tags = ", ".join(sorted(word_tags[word]))
    data.append((word, freq, tags))

# Create and display DataFrame
result_df2 = pd.DataFrame(data, columns=["Word", "Frequency", "Tags_op"])
result_df2.tail(10)

,Word,Frequency,Tags_op
990,सगळीं,131,"N_NN, QT_QTC, QT_QTF"
991,खाण,131,N_NN
992,क्षेत्र,131,"N_NN, N_NNP"
993,क्षेत्रांनी,131,"N_NN, QT_QTF"
994,घेवची,131,"V_VAUX_VF, V_VM_VF, V_VM_VNF"
995,पळयलें,131,"V_VAUX_VF, V_VM_VF, V_VM_VNF"
996,आडखळ,131,N_NN
997,काणी,130,N_NN
998,हिमाचल,130,N_NNP
999,४०,130,QT_QTC


In [10]:
result_df2.to_excel("df_ILCI_top_1000_wordfreq.xlsx", index=False)

In [31]:
# Convert 'Word' columns to sets
words_dhyas = set(result_df_dhyas["Word"])
words_anwani = set(result_df_anwani["Word"])
words_ilci = set(result_df_ilci_test["Word"])

# 1. Common words across all three
common_all = words_dhyas & words_anwani & words_ilci

# 2. Words only in dhyas
only_in_dhyas = words_dhyas - (words_anwani | words_ilci)

# 3. Words only in anwani
only_in_anwani = words_anwani - (words_dhyas | words_ilci)

# 4. Words only in ilci
only_in_ilci = words_ilci - (words_dhyas | words_anwani)

# 5. Union (total unique words from all three)
union_words = words_dhyas | words_anwani | words_ilci

# Optional: Common between any two
common_dhyas_anwani = words_dhyas & words_anwani
common_dhyas_ilci = words_dhyas & words_ilci
common_anwani_ilci = words_anwani & words_ilci

# Convert sets to DataFrames
df_common_all = pd.DataFrame({'Word': sorted(common_all)})
df_only_dhyas = pd.DataFrame({'Word': sorted(only_in_dhyas)})
df_only_anwani = pd.DataFrame({'Word': sorted(only_in_anwani)})
df_only_ilci = pd.DataFrame({'Word': sorted(only_in_ilci)})
df_union_words = pd.DataFrame({'Word': sorted(union_words)})

# Print stats
print("✅ Common words in all three:", len(df_common_all))
print("❌ Words only in dhyas:", len(df_only_dhyas))
print("❌ Words only in anwani:", len(df_only_anwani))
print("❌ Words only in ilci:", len(df_only_ilci))
print("🔄 Total unique words (union):", len(df_union_words))

✅ Common words in all three: 46
❌ Words only in dhyas: 367
❌ Words only in anwani: 411
❌ Words only in ilci: 500
🔄 Total unique words (union): 1442


In [33]:
print("\n🔹 Sample Common Words:")
df_common_all


🔹 Sample Common Words:


,Word
0,","
1,.
2,अशें
3,आनी
4,आयला
5,आशिल्ली
6,आशिल्लो
7,आसता
8,आसा
9,आसात


In [91]:
df_common_words.to_excel("df_common_words.xlsx", index=False)

In [34]:
print("\n🔸 Sample Only in Dhyas:")
df_only_dhyas




🔸 Sample Only in Dhyas:


,Word
0,1
1,10
2,100
3,105
4,1969
...,...
362,हाडिल्लें
363,हाताच्या
364,हायटॅक
365,होच


In [90]:
df_only_dhyas.to_excel("df_only_dhyas.xlsx", index=False)

In [35]:
print("\n🔸 Sample Only in Anwani:")
df_only_anwani


🔸 Sample Only in Anwani:


,Word
0,!
1,अम्मागेर
2,अय
3,अरे
4,अशा
...,...
406,हाताक
407,हुंपली
408,हुसको
409,हुसक्यान


In [89]:
df_only_anwani.to_excel("df_only_anwani.xlsx", index=False)

In [36]:
print("\n🔸 Sample Only in ILCI:")
df_only_ilci


🔸 Sample Only in ILCI:


,Word
0,-
1,418
2,5
3,Fatigue
4,`
...,...
495,२०११च्या
496,३१
497,६
498,‘


In [37]:
df_only_anwani.to_excel("df_only_ilci.xlsx", index=False)

In [87]:
print("\n🔄 Sample Union Words:")
df_union_words


🔄 Sample Union Words:


,Word
0,!
1,(
2,)
3,","
4,.
...,...
937,हेंच
938,हो
939,होच
940,ह्या


In [88]:
df_union_words.to_excel("df_union_words.xlsx", index=False)

In [39]:
# Step 1: Convert 'Word' columns to sets
words_dhyas = set(result_df_dhyas["Word"])
words_anwani = set(result_df_anwani["Word"])
words_ilci = set(result_df_ilci_test["Word"])

# Step 2: Compute sets
common_all = words_dhyas & words_anwani & words_ilci
only_in_dhyas = words_dhyas - (words_anwani | words_ilci)
only_in_anwani = words_anwani - (words_dhyas | words_ilci)
only_in_ilci = words_ilci - (words_dhyas | words_anwani)
union_words = words_dhyas | words_anwani | words_ilci

# Step 3: Create DataFrames with frequency for each category
def get_freq_df(word_set, source_df):
    return source_df[source_df["Word"].isin(word_set)][["Word", "Word_Freq"]].sort_values("Word_Freq", ascending=False)

df_common_all = get_freq_df(common_all, result_df_dhyas)  # From any DF, since word is in all
df_only_dhyas = get_freq_df(only_in_dhyas, result_df_dhyas)
df_only_anwani = get_freq_df(only_in_anwani, result_df_anwani)
df_only_ilci = get_freq_df(only_in_ilci, result_df_ilci_test)

# For union, merge all 3 and group by Word
combined_df = pd.concat([
    result_df_dhyas[["Word", "Word_Freq"]],
    result_df_anwani[["Word", "Word_Freq"]],
    result_df_ilci_test[["Word", "Word_Freq"]]
])
df_union_words = combined_df.groupby("Word", as_index=False)["Word_Freq"].sum().sort_values("Word_Freq", ascending=False)

# Step 4: Print statistics
print("✅ Common words in all three:", len(df_common_all))
print("❌ Words only in dhyas:", len(df_only_dhyas))
print("❌ Words only in anwani:", len(df_only_anwani))
print("❌ Words only in ilci:", len(df_only_ilci))
print("🔄 Total unique words (union):", len(df_union_words))

✅ Common words in all three: 46
❌ Words only in dhyas: 367
❌ Words only in anwani: 411
❌ Words only in ilci: 500
🔄 Total unique words (union): 1442


In [44]:
# Optional: Show top 5 from each
print("\n📌 Top common words:")
df_common_all.head()



📌 Top common words:


,Word,Word_Freq
0,.,93
1,",",19
2,आनी,16
3,आसता,12
6,आसात,10


In [45]:
df_common_all.to_excel("df_common_all_freq.xlsx", index=False)

In [46]:
print("\n📌 Top only in dhyas:")
df_only_dhyas.head()




📌 Top only in dhyas:


,Word,Word_Freq
4,गोंयांत,12
5,नुस्तें,12
7,पर्यटन,10
10,गोंय,8
12,सगल्यांत,7


In [47]:
df_only_dhyas.to_excel("df_only_dhyas_freq.xlsx", index=False)

In [48]:
print("\n📌 Top only in anwani:")
df_only_anwani.head()




📌 Top only in anwani:


,Word,Word_Freq
11,तुका,12
14,किदें,10
16,म्हणून,9
20,खंय,9
24,नाका,7


In [49]:
df_only_anwani.to_excel("df_only_anwani_freq.xlsx", index=False)

In [50]:
print("\n📌 Top only in ilci:")
df_only_ilci.head()


📌 Top only in ilci:


,Word,Word_Freq
6,-,6
8,कारण,5
26,वखदां,3
28,जर,3
32,बोकारो,3


In [51]:
df_only_ilci.to_excel("df_only_ilci_freq.xlsx", index=False)

In [52]:
print("\n📌 Total union of words:")
df_union_words.head()


📌 Total union of words:


,Word,Word_Freq
5,.,365
3,",",63
65,आनी,57
730,ना,52
22,?,47


In [53]:
df_union_words.to_excel("df_union_words_freq.xlsx", index=False)